# Coat64 Fast SWA submission

Export and validate the locally generated prediction CSV for the notebook-only competition submission flow.


In [ ]:
from pathlib import Path
import os
import pandas as pd

source = Path("/kaggle/input/rsna-knee-coat64-fast-swa-submission/submission.csv")
if not source.exists():
    raise FileNotFoundError(source)
pred = pd.read_csv(source, dtype={"StudyInstanceUID": str})

# Validate column order against the competition sample file, if mounted.
samples = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv"),
    Path("/kaggle/input/rsna-knee-abnormality-detection/sample_submission.csv"),
]
for sample_path in samples:
    if sample_path.exists():
        expected = list(pd.read_csv(sample_path, nrows=0).columns)
        if list(pred.columns) != expected:
            raise ValueError(f"Submission columns differ: {list(pred.columns)} != {expected}")
        break
if pred["StudyInstanceUID"].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID rows")
if not pred.iloc[:, 1:].notna().all().all():
    raise ValueError("Prediction values contain NaN")
output = Path("/kaggle/working/submission.csv")
pred.to_csv(output, index=False)
print(f"Wrote {output}: {len(pred)} studies x {len(pred.columns)} columns")
print(pred.head().to_string(index=False))
